# 04.01 — DDPM — 从零生成二维圆环

**Transformer-to-VLA · Chapter 04 / DDPM Fundamentals**

本 Notebook 不生成图像，只使用二维点学习 DDPM 的核心机制：**数据分布 → 随机加噪 → 噪声预测 → 从纯噪声反向采样**。

**学习成果**：理解 `beta / alpha / alpha_bar`、为何随机采样时间步、噪声预测 MSE、如何恢复二维圆环。所有单元均可在 Google Colab 的 CPU / GPU 上运行，不依赖外部文件。

运行方式：`Runtime → Run all`；若 CPU 时间有限，可先减少 `TRAIN_STEPS`。

## 1. 环境与可复现性
仅使用 Colab 常见的 `torch`、`numpy`、`matplotlib`。设置随机种子，自动选择 GPU 或 CPU。

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", device)


## 2. 构造二维圆环数据

随机角度 $\theta\sim U(0,2\pi)$，构造

$$
\mathbf{x}_0=[\cos\theta,\sin\theta]^\top+\boldsymbol{\eta},\qquad \boldsymbol{\eta}\sim\mathcal N(0,0.05^2I).
$$

每条数据只是一个二维坐标，形状 `[B, 2]`。

In [ ]:
def sample_data(batch_size, device=device):
    theta = 2 * math.pi * torch.rand(batch_size, device=device)
    x = torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)
    return x + 0.05 * torch.randn_like(x)

real = sample_data(2000).cpu().numpy()
plt.figure(figsize=(4.8, 4.8))
plt.scatter(real[:, 0], real[:, 1], s=5, alpha=0.5)
plt.title("Training data: noisy 2D ring")
plt.axis("equal"); plt.grid(alpha=0.2); plt.show()
print("batch shape:", sample_data(32).shape)

## 3. 定义前向扩散过程

使用原实验的 $T=100$ 和线性噪声调度：

$$
\beta_t\in[10^{-4},0.02],\quad\alpha_t=1-\beta_t,\quad\bar\alpha_t=\prod_{s=1}^{t}\alpha_s.
$$

直接采样任意时间步（不需要逐步执行前向扩散）：

$$
\mathbf x_t=\sqrt{\bar\alpha_t}\mathbf x_0+\sqrt{1-\bar\alpha_t}\boldsymbol\epsilon,\qquad\boldsymbol\epsilon\sim\mathcal N(0,I).
$$

**代码下标约定**：`t=0` 表示数学公式中的第 1 个扩散步（而非完全无噪声的 $x_0$）。

In [ ]:
T = 100
beta = torch.linspace(1e-4, 0.02, T, device=device)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    scale_x = alpha_bar[t].sqrt().unsqueeze(-1)
    scale_noise = (1.0 - alpha_bar[t]).sqrt().unsqueeze(-1)
    return scale_x * x0 + scale_noise * noise

x0 = sample_data(1000)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, t_value in zip(axes, [0, 19, 39, 69, 99]):
    xt = q_sample(x0, torch.full((len(x0),), t_value, device=device, dtype=torch.long))
    points = xt.cpu().numpy()
    ax.scatter(points[:, 0], points[:, 1], s=3, alpha=0.35)
    ax.set(title=f"t={t_value+1}", xlim=(-3, 3), ylim=(-3, 3), aspect="equal")
plt.suptitle("Forward diffusion (direct sampling)"); plt.tight_layout(); plt.show()

## 4. 从零实现噪声预测网络

输入：带噪二维点 $\mathbf x_t\in\mathbb R^2$ 与归一化时间标量 $t/T$。输出：预测噪声 $\epsilon_\theta(\mathbf x_t,t)\in\mathbb R^2$。

第一版故意用最简单的 MLP：`[x, y, t/T] → 128 → 128 → [epsilon_x, epsilon_y]`，以便理解 DDPM 而不是网络架构。

In [ ]:
class NoisePredictor(nn.Module):
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x_t, t):
        t_normalized = t.float().unsqueeze(-1) / T
        return self.net(torch.cat([x_t, t_normalized], dim=-1))

model = NoisePredictor().to(device)
print(model)
print("output shape:", model(torch.randn(8, 2, device=device), torch.randint(T, (8,), device=device)).shape)

## 5. 训练：预测注入的高斯噪声

随机选取每个样本的时间步，构造 $\mathbf x_t$，最小化

$$
\mathcal L=\mathbb E_{\mathbf x_0,t,\epsilon}\big[\|\epsilon-\epsilon_\theta(\mathbf x_t,t)\|_2^2\big].
$$

原理演示默认训练 3000 步；若希望 CPU 更快完成，可改为 800–1500 步。这里使用 Adam 和 MSE。

In [ ]:
TRAIN_STEPS = 3000        # Colab CPU 可先改为 1000
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_history = []
model.train()
for step in range(TRAIN_STEPS):
    x0 = sample_data(BATCH_SIZE)
    t = torch.randint(0, T, (BATCH_SIZE,), device=device)
    noise = torch.randn_like(x0)
    x_t = q_sample(x0, t, noise)
    prediction = model(x_t, t)
    loss = torch.nn.functional.mse_loss(prediction, noise)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if (step + 1) % max(1, TRAIN_STEPS // 5) == 0:
        print(f"step {step + 1:5d}/{TRAIN_STEPS} | loss {loss.item():.4f}")

plt.figure(figsize=(7, 3))
plt.plot(loss_history, linewidth=0.8)
plt.xlabel("Training step"); plt.ylabel("Noise prediction MSE")
plt.title("Training loss"); plt.grid(alpha=0.2); plt.show()

## 6. 反向采样：从高斯噪声恢复圆环

采用 DDPM 常见的反向更新式：

$$
\mu_\theta(\mathbf x_t,t)=\frac{1}{\sqrt{\alpha_t}}\left(\mathbf x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta(\mathbf x_t,t)\right).
$$

对 $t>1$ 加入方差为 $\beta_t$ 的随机噪声；最后一步不添加。**这是简化的 fixed-large variance 选择**，第二个实验会使用更精确的 posterior variance。

In [ ]:
@torch.no_grad()
def sample_ddpm(n_samples=2000, capture_steps=(99, 79, 59, 39, 19, 0)):
    model.eval()
    x = torch.randn(n_samples, 2, device=device)
    snapshots = {"initial noise": x.cpu().numpy().copy()}
    for step in reversed(range(T)):
        t = torch.full((n_samples,), step, device=device, dtype=torch.long)
        noise_pred = model(x, t)
        mean = (x - beta[step] / torch.sqrt(1-alpha_bar[step]) * noise_pred) / torch.sqrt(alpha[step])
        if step > 0:
            x = mean + torch.sqrt(beta[step]) * torch.randn_like(x)
        else:
            x = mean
        if step in capture_steps:
            snapshots[f"after step {step+1}"] = x.cpu().numpy().copy()
    return x.cpu().numpy(), snapshots

generated, snapshots = sample_ddpm()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, points, name in zip(axes, [real, generated], ["Original 2D ring", "DDPM generated"]):
    ax.scatter(points[:, 0], points[:, 1], s=4, alpha=0.5)
    ax.set(title=name, xlim=(-3, 3), ylim=(-3, 3), aspect="equal")
plt.tight_layout(); plt.show()

## 7. 可视化逐步反向去噪
观察从随机二维高斯点云，逐渐凝聚到圆环的过程。注意：**每张图显示的是所有粒子的分布，不是一个二维粒子的运动轨迹。**

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (label, points) in zip(axes.flat, snapshots.items()):
    ax.scatter(points[:, 0], points[:, 1], s=2, alpha=0.3)
    ax.set(title=label, xlim=(-3, 3), ylim=(-3, 3), aspect="equal")
for ax in axes.flat[len(snapshots):]: ax.axis("off")
plt.tight_layout(); plt.show()

## 8. 动手实验与知识检查

1. 将 `T` 改为 50 / 200，并**重新从噪声调度开始运行**：扩散速度有何变化？
2. 将训练数据的 `0.05` 改为 `0.2`：模型生成的圆环厚度如何变化？
3. 将 `TRAIN_STEPS` 改为 300、1000、3000：训练不足时会发生什么？
4. 想一想：为什么训练时可以直接采样任意 $x_t$，但生成时要从最后一个时间步反向迭代？

**通往机器人策略**：现在生成的是 $x\in\mathbb R^2$；之后可以把每个样本替换成一段 $A\in\mathbb R^{H\times d_a}$ 的动作序列。训练与逆向采样框架仍可复用。